# Example 2: HiggsML Challenge with Convolutional Networks

In [1]:
from pathlib import Path
import urllib.request
from typing import Literal
import math
import pandas as pd
import numpy as np
import torch
from torch import Tensor
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torch.optim as optim
import matplotlib.pyplot as plt
import tqdm

## Define some utilitiy functions

In [3]:
def standardize(x: Tensor) -> Tensor:
    """Standardizes tensor along feature dimension (zero mean, unit variance)
    """
    dim = tuple(range(x.ndim - 1))
    return (x - x.mean(dim=dim)) / x.std(dim=dim)

def to_numpy(tensor):
    """Detaches tensor from computation graph and moves it to CPU for NumPy conversion
    """
    return tensor.detach().cpu().numpy()

def ams_score(y_true: torch.Tensor, y_score: torch.Tensor, weight: torch.Tensor, threshold: float = 0.5, br: float = 10.0):
    """Function to compute AMS (approximate median significance)
    """
    y_pred = y_score > threshold
    s = weight[(y_true == 1) & (y_pred == 1)].sum()
    b = weight[(y_true == 0) & (y_pred == 1)].sum()
    return torch.sqrt(2 * ((s + b + br) * torch.log(1.0 + s / (b + br)) - s))

## Dataset Class

In [12]:
class HiggsDataset(Dataset):
    """Custom PyTorch dataset for handling Higgs Boson data."""

    der_column_list = [
        'DER_mass_MMC',
        'DER_mass_transverse_met_lep',
        'DER_mass_vis',
        'DER_pt_h',
        'DER_deltaeta_jet_jet',
        'DER_mass_jet_jet',
        'DER_prodeta_jet_jet',
        'DER_deltar_tau_lep',
        'DER_pt_tot',
        'DER_sum_pt',
        'DER_pt_ratio_lep_tau',
        'DER_met_phi_centrality',
        'DER_lep_eta_centrality',
    ]
    

    long_tail_der_feature_list = [
        'PRI_tau_pt', 'PRI_lep_pt', 'PRI_met', 'PRI_met_sumet', 'PRI_jet_all_pt',
        'PRI_jet_leading_pt', 'PRI_jet_subleading_pt', 'DER_mass_MMC',
        'DER_mass_transverse_met_lep', 'DER_mass_vis', 'DER_pt_h',
        'DER_mass_jet_jet', 'DER_pt_tot', 'DER_sum_pt', 'DER_pt_ratio_lep_tau',
    ]

    def __init__(self, path, subset: Literal['t', 'b', 'v', 'u'] = 't'):
        df = pd.read_csv(path)

        # Loads only the specified subset (train, validation, etc.)
        df = df[df['KaggleSet'].eq(subset)]

        lep = self.make_obj(df, 'lep')
        tau = self.make_obj(df, 'tau')
        met = self.make_obj(df, 'met')
        
        jet0 = self.make_obj(df, 'jet_leading')
        jet1 = self.make_obj(df, 'jet_subleading')
        
        jet = torch.stack([jet0, jet1], dim=1)
        # Masks padded jets with unphysical values (like -999).
        jet_pad_mask = jet[..., 0] == -999
        jet.masked_fill_(jet_pad_mask[..., None], 0)

        # Loads and processes DER (derived) features, applying log transform
        der = {key: torch.from_numpy(df[key].to_numpy(np.float32)) for key in self.der_column_list}
        # Imputing
        for value in der.values():
            value[value == -999] = 0
            
        for key in der:
            if key in self.long_tail_der_feature_list:
                der[key] = der[key].log1p()

        der = [der[key] for key in self.der_column_list]
        der = torch.stack(der, dim=1)
        
        # preprocessing
        #der = standardize(der)
        lep = standardize(lep)
        tau = standardize(tau)
        met = standardize(met)
        jet = standardize(jet)
        der = standardize(der)

        target = torch.from_numpy(df['Label'].eq('s').to_numpy(np.int64))
        event_weight = torch.from_numpy(df['KaggleWeight'].to_numpy(np.float32))

        self.data = dict(
            der=der,
            tau=tau,
            lep=lep,
            met=met,
            jet=jet,
            jet_pad_mask=jet_pad_mask,
            target=target,
            event_weight=event_weight,
        )
            
    def __len__(self) -> int:
        return len(self.data['target'])
    
    def __getitem__(self, index: int):
        return {key: value[index] for key, value in self.data.items()}

    def make_obj(self, df: pd.DataFrame, name: str) -> np.ndarray:
        """Converts PRI (primitive) features into 3-vector representations like (px, py, eta or log(sumET))"""
        if name == 'met':
            pt = df['PRI_met'].to_numpy(np.float32)
            phi = df['PRI_met_phi'].to_numpy(np.float32)
            sumet = df['PRI_met_sumet'].to_numpy(np.float32)
            
            pt = torch.from_numpy(pt)
            phi = torch.from_numpy(phi)
            sumet = torch.from_numpy(sumet)

            # Converts raw kinematic features into Cartesian components
            px = pt * phi.cos()
            py = pt * phi.sin()
            sumet_log1p = sumet.log1p()
    
            obj = torch.stack([px, py, sumet_log1p], dim=1)
        else:
            pt = df[f'PRI_{name}_pt'].to_numpy(np.float32)
            eta = df[f'PRI_{name}_eta'].to_numpy(np.float32)
            phi = df[f'PRI_{name}_phi'].to_numpy(np.float32)

            pt = torch.from_numpy(pt)
            eta = torch.from_numpy(eta)
            phi = torch.from_numpy(phi)

            # Converts raw kinematic features into Cartesian components
            px = pt * torch.cos(phi)
            py = pt * torch.sin(phi)
            
            obj = torch.stack([px, py, eta], dim=1)
        return obj

## Constructs PyTorch Datasets and DataLoaders for batching.

In [13]:
dataset_file_path = Path('./data/dataset.csv.gz')
if not dataset_file_path.exists():
    raise FileNotFoundError(dataset_file_path)

In [14]:
train_set = HiggsDataset(dataset_file_path, subset='t')
val_set = HiggsDataset(dataset_file_path, subset='b')

In [ ]:
example = train_set[0]

In [16]:
example

{'der': tensor([ 0.5326,  0.4047,  0.6608, -0.0269,  0.1477,  1.2802,  1.4770,  0.8825,
          1.1611,  0.6919,  0.3527,  1.2771,  0.2242]),
 'tau': tensor([0.9585, 0.3946, 0.8467]),
 'lep': tensor([-1.0476, -0.9770,  1.8123]),
 'met': tensor([ 0.4291, -0.1181,  0.6455]),
 'jet': tensor([[ 1.2342,  0.9547,  1.1205],
         [ 1.0393, -0.2942,  1.1187]]),
 'jet_pad_mask': tensor([False, False]),
 'target': tensor(1),
 'event_weight': tensor(0.0027)}

In [ ]:
len(train_set), len(val_set)

(250000, 100000)

In [8]:
train_loader = DataLoader(
    dataset=train_set,
    batch_size=128,
    shuffle=True,
    drop_last=True,
)


val_loader = DataLoader(
    dataset=val_set,
    batch_size=128,
    shuffle=False,
    drop_last=False,
)

In [40]:
class Model(nn.Module):

    """Deep neural network with primitive object encoders + derived (engineered) feature encoder + classifier."""

    def __init__(
        self,
        der_dim: int = 13,
        tau_dim: int = 3,
        lep_dim: int = 3,
        met_dim: int = 3,
        jet_dim: int = 3,
        hidden_dim: int = 64,
    ) -> None:
        super().__init__()

        self.tau_proj = nn.Linear(in_features=tau_dim, out_features=hidden_dim)
        self.lep_proj = nn.Linear(in_features=lep_dim, out_features=hidden_dim)
        self.met_proj = nn.Linear(in_features=met_dim, out_features=hidden_dim)
        self.jet_proj = nn.Linear(in_features=jet_dim, out_features=hidden_dim)

        # 1D CNN block for encoding combined object features (lep, tau, met, jets).
        self.pri_encoder = nn.Sequential(
            nn.Conv1d(in_channels=hidden_dim, out_channels=hidden_dim, kernel_size=1),
            nn.BatchNorm1d(num_features=hidden_dim),
            nn.LeakyReLU(),
            nn.Conv1d(in_channels=hidden_dim, out_channels=hidden_dim, kernel_size=3),
            nn.BatchNorm1d(num_features=hidden_dim),
            nn.LeakyReLU(),
            nn.Conv1d(in_channels=hidden_dim, out_channels=hidden_dim, kernel_size=3),
            nn.BatchNorm1d(num_features=hidden_dim),
            nn.LeakyReLU(),
        )

        # MLP for engineered DER features.
        self.der_encoder = nn.Sequential(
            nn.Linear(in_features=der_dim, out_features=hidden_dim),
            nn.BatchNorm1d(num_features=hidden_dim),
            nn.LeakyReLU(),
            nn.Linear(in_features=hidden_dim, out_features=hidden_dim),
            nn.BatchNorm1d(num_features=hidden_dim),
            nn.LeakyReLU(),
        )

        # MLP for binary classification.
        self.classification_head = nn.Sequential(
            nn.Linear(in_features=2 * hidden_dim, out_features=hidden_dim),
            nn.BatchNorm1d(num_features=hidden_dim),
            nn.GELU(),
            nn.Linear(in_features=hidden_dim, out_features=hidden_dim),
            nn.BatchNorm1d(num_features=hidden_dim),
            nn.GELU(),
            nn.Linear(in_features=hidden_dim, out_features=hidden_dim),
            nn.BatchNorm1d(num_features=hidden_dim),
            nn.GELU(),
            nn.Linear(in_features=hidden_dim, out_features=1),
        )

    def forward(
        self,
        der: Tensor,
        tau: Tensor,
        lep: Tensor,
        met: Tensor,
        jet: Tensor,
        jet_pad_mask: Tensor,
    ) -> Tensor:
        """
        """
        der = self.der_encoder(der)

        tau = self.tau_proj(tau)
        lep = self.lep_proj(lep)
        met = self.met_proj(met)
        jet = self.jet_proj(jet)

        jet = jet.masked_fill(jet_pad_mask[..., None], 0)
        
        tau = tau.unsqueeze(dim=-1)
        lep = lep.unsqueeze(dim=-1)
        met = met.unsqueeze(dim=-1)
        jet = jet.permute(0, 2, 1).contiguous()

        pri = torch.concatenate([tau, lep, met, jet], dim=2)
        
        pri = self.pri_encoder(pri)
        pri = pri.squeeze()

        x = torch.cat([der, pri], dim=-1)
        
        x = self.classification_head(x)
        x = x.squeeze()
        return x

## Set up model, weighted binary loss (class imbalance), and optimizer.


In [ ]:
model = Model(hidden_dim=128)

In [11]:
_, (train_num_bkg, train_num_sig) = train_set.data['target'].unique(return_counts=True)
pos_weight = train_num_bkg / train_num_sig
print(f'{pos_weight}')

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [12]:
optimizer = optim.AdamW(params=model.parameters(), lr=3e-4)

In [13]:
num_params = sum(each.numel() for each in model.parameters())
print(f'{num_params:_}')

203_521


In [14]:
device = torch.device('cuda:1')

In [15]:
model = model.to(device)
criterion = criterion.to(device)

## Iterates over epochs, training and validating. Tracks loss and early stopping.


In [ ]:
max_epochs = 5
# max_epochs = 100 # for your homework
threshold = 0.5

In [ ]:
best_val_loss = float('inf')
early_stop_patience = 3
early_stop_counter = 0
best_ckpt_path = Path('higgsml-convnet-best.pth')

In [ ]:
for epoch in range(0, max_epochs + 1):
    if epoch > 0:
        model.train()
        for batch in tqdm.tqdm(train_loader, desc='training'):
            batch = {key: value.to(device) for key, value in batch.items()}
            optimizer.zero_grad()
            logits = model(
                der=batch['der'],
                tau=batch['tau'],
                lep=batch['lep'],
                met=batch['met'],
                jet=batch['jet'],
                jet_pad_mask=batch['jet_pad_mask'],
            )
            loss = criterion(
                input=logits, 
                target=batch['target'].to(logits.dtype),
            )
            loss.backward()
            optimizer.step()

    
    with torch.inference_mode():
        # validation
        model.eval()

        val_loss = 0
        val_total = 0
        val_correct = 0

        for batch in tqdm.tqdm(val_loader, desc='validation'):
            batch = {key: value.to(device) for key, value in batch.items()}
            logits = model(
                der=batch['der'],
                tau=batch['tau'],
                lep=batch['lep'],
                met=batch['met'],
                jet=batch['jet'],
                jet_pad_mask=batch['jet_pad_mask'],
            )
            score = logits.sigmoid()
            prediction = score.gt(threshold).long()


            y_true = batch['target']
            loss = criterion(input=logits, target=y_true.to(logits.dtype))

            val_loss += len(y_true) * loss.item()
            val_total += len(y_true)
            val_correct += prediction.eq(y_true).sum().item()
    
        val_loss /= val_total
        val_acc = val_correct / val_total
        print(f'{epoch=: >6d}: Loss={val_loss:.3f},  Accuracy={100 * val_acc:.2f} %')

    # Early stopping check
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        early_stop_counter = 0
        torch.save(model.state_dict(), best_ckpt_path)
    else:
        early_stop_counter += 1
        if early_stop_counter >= early_stop_patience:
            print("Early stopping triggered")
            break

# Evaluate

In [29]:
with torch.inference_mode():
    model.eval()

    y_true_cat = []
    y_score_cat = []
    event_weight_cat = []

    for batch in tqdm.tqdm(val_loader, desc='validation'):
        batch = {key: value.to(device) for key, value in batch.items()}
        y_logits = model(
            der=batch['der'],
            tau=batch['tau'],
            lep=batch['lep'],
            met=batch['met'],
            jet=batch['jet'],
            jet_pad_mask=batch['jet_pad_mask'],
        )
        y_score = y_logits.sigmoid()

        y_score_cat.append(y_score.cpu())
        y_true_cat.append(batch['target'].cpu())
        event_weight_cat.append(batch['event_weight'].cpu())

validation: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 782/782 [00:03<00:00, 196.77it/s]


In [30]:
y_true = torch.cat(y_true_cat)
y_score = torch.cat(y_score_cat)
event_weight = torch.cat(event_weight_cat)

### Visualizes AMS vs threshold curve

In [ ]:
ams_threshold_arr = torch.linspace(0, 1, 20)

In [36]:
ams_arr = [
    ams_score(
        y_true=y_true,
        y_score=y_score, 
        weight=event_weight,
        threshold=threshold.item(),
    )
    for threshold in threshold_arr
]

In [ ]:
# Find best threshold and max AMS
best_idx = np.argmax(ams_arr)
max_ams = ams_arr[best_idx]

# Plot AMS curve
fig, ax = plt.subplots()
ax.plot(ams_threshold_arr, ams_arr, lw=2)
ax.axvline(ams_threshold_arr[best_idx], color='gray', ls=':')
ax.plot([ams_threshold_arr[best_idx]], [ams_arr[best_idx]], ls='', marker='*', color='red', markersize=20, label=f'Max AMS: {max_ams:.3f}')
ax.set_xlabel('Threshold')
ax.set_ylabel('AMS')
ax.legend()
ax.grid()